In [8]:
from tqdm import tqdm
import json
import os
import zarr
import cv2
import numpy as np
from utils.config import Config

In [9]:
def get_or_compute_normalization(config, volume, mask):
    """Retrieve or compute global mean, std, min, and max for normalization."""
    segment_id = config.data.segment_id
    cache = _load_normalization_cache()

    if str(segment_id) in cache:
        print(f"[INFO] Using cached normalization for segment {segment_id}")
        return cache[str(segment_id)]["mean"], cache[str(segment_id)]["std"], cache[str(segment_id)]["min"], cache[str(segment_id)]["max"]

    print(f"[INFO] Computing normalization for segment {segment_id}")
    total_sum, total_squared_sum, total_count = 0.0, 0.0, 0
    global_min, global_max = float('inf'), float('-inf')

    for z in tqdm(range(volume.shape[0])):
        for y in range(0, volume.shape[1], 1024):
            for x in range(0, volume.shape[2], 1024):
                chunk = volume[z, y:y+1024, x:x+1024]
                mask_chunk = mask[y:y+1024, x:x+1024]
                valid_pixels = chunk[mask_chunk > 0]
                if valid_pixels.size == 0:
                    continue

                total_sum += np.sum(valid_pixels, dtype=np.float64)
                total_squared_sum += np.sum(valid_pixels.astype(np.float64) ** 2, dtype=np.float64)
                total_count += valid_pixels.size

    if total_count == 0:
        raise ValueError("No valid pixels found in the dataset.")

    # Compute global mean and std
    global_mean = total_sum / total_count
    mean_of_squares = total_squared_sum / total_count
    square_of_mean = global_mean ** 2
    variance = max(mean_of_squares - square_of_mean, 0)
    global_std = np.sqrt(variance)

    # Compute global min and max after normalization
    for z in tqdm(range(volume.shape[0])):
        for y in range(0, volume.shape[1], 1024):
            for x in range(0, volume.shape[2], 1024):
                chunk = volume[z, y:y+1024, x:x+1024]
                mask_chunk = mask[y:y+1024, x:x+1024]
                valid_pixels = chunk[mask_chunk > 0]
                if valid_pixels.size == 0:
                    continue

                # Normalize the valid pixels
                normalized_pixels = (valid_pixels.astype(np.float64) - global_mean) / global_std

                # Update global min and max
                global_min = min(global_min, normalized_pixels.min())
                global_max = max(global_max, normalized_pixels.max())

    print(f"Final Statistics:")
    print(f"  Global Mean: {global_mean:.6f}")
    print(f"  Global Std: {global_std:.6f}")
    print(f"  Global Min (after normalization): {global_min:.6f}")
    print(f"  Global Max (after normalization): {global_max:.6f}")

    _update_normalization_cache(segment_id, global_mean, global_std, global_min, global_max)
    return global_mean, global_std, global_min, global_max


def _load_normalization_cache():
    """Load normalization cache from file."""
    if not os.path.exists("./normalization_cache.json"):
        print(f"[INFO] No cache found.")
        return {}
    with open("./normalization_cache.json", "r") as f:
        print(f"[INFO] Cache found, loading...")
        return json.load(f)

def _update_normalization_cache(segment_id, mean, std, min_val, max_val):
    """Update normalization cache with new values."""
    cache = _load_normalization_cache()
    cache[str(segment_id)] = {
        "mean": float(mean),  # Convert to float
        "std": float(std),    # Convert to float
        "min": float(min_val),  # Convert to float
        "max": float(max_val)   # Convert to float
    }

    try:
        with open("./normalization_cache.json", "w") as f:
            json.dump(cache, f, indent=4)  # Use indent for readability
            f.flush()  # Ensure data is written to disk
        print(f"[INFO] Updated normalization cache for segment {segment_id} with mean={mean:.6f}, std={std:.6f}, min={min_val:.6f}, max={max_val:.6f}")
    except Exception as e:
        print(f"[ERROR] Failed to update normalization cache: {e}")

In [10]:
volume = zarr.open("/media/jeff/Seagate/vesuvius-3dstreamer/20230827161847_singlethreaded.zarr", mode='r')
mask = cv2.imread("/media/jeff/Seagate/vesuvius/masks/20230827161847.png", cv2.IMREAD_GRAYSCALE) / 255.0
config = Config()

In [13]:
global_mean, global_std, global_min, global_max = get_or_compute_normalization(config, volume, mask)
print(f"mean: {global_mean}")
print(f"std: {global_std}")
print(f"min: {global_min}")
print(f"max: {global_max}")

[INFO] Cache found, loading...
[INFO] Computing normalization for segment 20230827161847


100%|██████████| 64/64 [01:49<00:00,  1.71s/it]

Final Statistics:
  Global Mean: 32177.302037
  Global Std: 11522.682260
  Global Min (after normalization): -2.792518
  Global Max (after normalization): 2.894959
[INFO] Cache found, loading...
[INFO] Updated normalization cache for segment 20230827161847 with mean=32177.302037, std=11522.682260, min=-2.792518, max=2.894959
mean: 32177.302037344252
std: 11522.682259840638
min: -2.7925183834574705
max: 2.8949594556569065
